<a href="https://colab.research.google.com/github/MarkAntonio/Topic-Modeling-Using-Transformers/blob/main/topic_modelling_using_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ! python -m spacy download en_core_web_lg
# ! pip install unidecode
# ! pip install pyLDAvis


2023-12-08 23:19:32.229691: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-12-08 23:19:32.229765: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-12-08 23:19:32.229842: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-12-08 23:19:32.240533: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-12-08 23:19:33.825429: W tensorflow/compiler/

In [ ]:
# !pip install pandas==1.5.3
# !pip install pandas
!pip install pandas==2

### Bibliotecas

In [ ]:
import os
import pandas as pd
import re

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from unidecode import unidecode

import gensim
import pyLDAvis.gensim
import pickle
import pyLDAvis

import spacy
nlp = spacy.load('en_core_web_lg')

nltk.download('punkt')
nltk.download('stopwords')

/usr/local/lib/python3.10/dist-packages/tensorflow/python/framework/dtypes.py:35: DeprecationWarning: ml_dtypes.float8_e4m3b11 is deprecated. Use ml_dtypes.float8_e4m3b11fnuz
  from tensorflow.tsl.python.lib.core import pywrap_ml_dtypes
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
stop_words = set(stopwords.words('english'))

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Base de Dados

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Modelagem de tópicos usando Transformer/Cópia de Restaurant_Reviews.tsv', sep='\t')

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
df

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1
...,...,...
995,I think food should have flavor and texture an...,0
996,Appetite instantly gone.,0
997,Overall I was not impressed and would not go b...,0
998,"The whole experience was underwhelming, and I ...",0


### Limpeza dos dados

In [ ]:
## Retirando caracteres especiais e passando texto para minúsculo
def remove_special_tokens(text):

    formated_text = re.sub(r'[^a-zA-Z\s]+', ' ', text)

    return formated_text.lower()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
## retirada de espaços em brancos
def remove_space(text):
  # clean_text = re.sub('\s+', ' ', text)
  clean_text = re.sub('\s{2,}', ' ', text)
  return clean_text

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<>:4: DeprecationWarning: invalid escape sequence '\s'
<>:4: DeprecationWarning: invalid escape sequence '\s'
<ipython-input-32-8f30bd3c20f1>:4: DeprecationWarning: invalid escape sequence '\s'
  clean_text = re.sub('\s{2,}', ' ', text)


In [ ]:
## Retirando números
def remove_numbers(text):
  clean_text = re.sub('r[0-9]','', text)
  return clean_text

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
## Tokenizacao
# separa toda a frase pelos espaços e tranforma em uma lista de strings
def tokenize_text(text):

    result = ' '.join(word_tokenize(text))

    return result

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
## Lematizacao
"""
é o processo para reduzir uma palavra para sua raiz ou para uma forma padrão
Amigo, amiga, amigão -> amigo
Eat, eating, eaten, ate -> eat
"""
def lematize(doc):
    doc = nlp(doc)
    sentence = []
    for token in doc:
      sentence.append(token.lemma_)

    formattedText = ""
    for word in sentence:
        formattedText = formattedText + " " +word

    return formattedText

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
stop_words

In [ ]:
# palavras que posuem um alto grau de repetição mas que não acrescentam informação (a, or, and, an)
def remove_stop_words(example_sent):

    word_tokens = word_tokenize(example_sent)


    filtered_sentence = []

    for w in word_tokens:
        if w.lower() not in stop_words:
            filtered_sentence.append(w)

    formattedText = ""

    for word in filtered_sentence:

            formattedText = formattedText + " " +word

    formattedText = formattedText.lstrip()

    formattedText = formattedText.rstrip()

    return formattedText.lower()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
raw_text = df.Review.tolist()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
raw_text[:10]

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


['Wow... Loved this place.',
 'Crust is not good.',
 'Not tasty and the texture was just nasty.',
 'Stopped by during the late May bank holiday off Rick Steve recommendation and loved it.',
 'The selection on the menu was great and so were the prices.',
 'Now I am getting angry and I want my damn pho.',
 "Honeslty it didn't taste THAT fresh.)",
 'The potatoes were like rubber and you could tell they had been made up ahead of time being kept under a warmer.',
 'The fries were great too.',
 'A great touch.']

In [ ]:
from copy import copy
texts = copy(raw_text)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
remove_special_tokens(texts[0])

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


'wow  loved this place '

In [ ]:
word_tokenize(texts[0])

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


['Wow', '...', 'Loved', 'this', 'place', '.']

In [ ]:
lematize(texts[3])

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


' stop by during the late May bank holiday off Rick Steve recommendation and love it .'

### Juntando toda a parte de pré-processamento

In [ ]:
## Aqui juntaremos todas as partes do processamento
def pipe_clean_data(file_text):
    # print('cleaning text')
    try:

        final_text = file_text


        #removing spaces
        final_text = remove_space(final_text)
        final_text = final_text.lstrip()
        final_text = final_text.rstrip()

        clean_sentence = unidecode(final_text)
        clean_sentence = remove_special_tokens(clean_sentence)
        clean_sentence = tokenize_text(clean_sentence)
        clean_sentence = remove_stop_words(clean_sentence)


        clean_sentence = lematize(clean_sentence)

        return clean_sentence

    except Exception as e:
        print(f'Text to clean:{file_text}')
        print(e)
        print(type(file_text), len(file_text))
        # break


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
clean_text = []
for text in texts:
  clean_text.append(pipe_clean_data(text))

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Text to clean:Wow... Loved this place.
name 'stop_words' is not defined
<class 'str'> 24
Text to clean:Crust is not good.
name 'stop_words' is not defined
<class 'str'> 18
Text to clean:Not tasty and the texture was just nasty.
name 'stop_words' is not defined
<class 'str'> 41
Text to clean:Stopped by during the late May bank holiday off Rick Steve recommendation and loved it.
name 'stop_words' is not defined
<class 'str'> 87
Text to clean:The selection on the menu was great and so were the prices.
name 'stop_words' is not defined
<class 'str'> 59
Text to clean:Now I am getting angry and I want my damn pho.
name 'stop_words' is not defined
<class 'str'> 46
Text to clean:Honeslty it didn't taste THAT fresh.)
name 'stop_words' is not defined
<class 'str'> 37
Text to clean:The potatoes were like rubber and you could tell they had been made up ahead of time being kept under a warmer.
name 'stop_words' is not defined
<class 'str'> 111
Text to clean:The fries were great too.
name 'stop_words

### Amostragem do texto por palavras

In [ ]:
from collections import Counter
data_words = []
for item in clean_text:
    data_words+=item.split(' ')

cnt_words = Counter(data_words)
cnt_words.most_common(30)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


[('', 1000),
 ('food', 127),
 ('good', 121),
 ('place', 112),
 ('service', 87),
 ('go', 76),
 ('great', 71),
 ('back', 61),
 ('time', 54),
 ('like', 51),
 ('get', 42),
 ('come', 38),
 ('really', 36),
 ('well', 35),
 ('eat', 34),
 ('bad', 31),
 ('would', 29),
 ('wait', 29),
 ('love', 28),
 ('ever', 28),
 ('restaurant', 28),
 ('order', 28),
 ('also', 27),
 ('friendly', 27),
 ('nice', 27),
 ('one', 27),
 ('make', 26),
 ('say', 26),
 ('never', 26),
 ('amazing', 24)]

### Representação de dados de Texto

In [ ]:
## Aqui vamos o CountVector para criar uma matrix de Frequência de Palavras (BoW)
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer='word', ngram_range=(1, 2))
X2 = vectorizer.fit_transform(clean_text)
vectorizer.get_feature_names_out()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


NameError: ignored

In [ ]:
X2.toarray().shape

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


(1000, 5664)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


### Aplicando a técnica LDA

In [ ]:
tokens_by_sentence = [word_tokenize(text) for text in clean_text]

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
print(tokens_by_sentence)

[['wow', 'loved', 'place'], ['crust', 'good'], ['tasty', 'texture', 'nasty'], ['stop', 'late', 'may', 'bank', 'holiday', 'rick', 'steve', 'recommendation', 'love'], ['selection', 'menu', 'great', 'price'], ['get', 'angry', 'want', 'damn', 'pho'], ['honeslty', 'taste', 'fresh'], ['potato', 'like', 'rubber', 'could', 'tell', 'make', 'ahead', 'time', 'keep', 'warm'], ['fry', 'great'], ['great', 'touch'], ['service', 'prompt'], ['would', 'go', 'back'], ['cashier', 'care', 'ever', 'say', 'still', 'end', 'wayyy', 'overprice'], ['try', 'cape', 'cod', 'ravoli', 'chicken', 'cranberry', 'mmmm'], ['disgust', 'pretty', 'sure', 'human', 'hair'], ['shocked', 'sign', 'indicate', 'cash'], ['highly', 'recommend'], ['waitress', 'little', 'slow', 'service'], ['place', 'worth', 'time', 'let', 'alone', 'vegas'], ['like'], ['burritto', 'blah'], ['food', 'amazing'], ['service', 'also', 'cute'], ['could', 'care', 'less', 'interior', 'beautiful'], ['perform'], ['right', 'red', 'velvet', 'cake', 'ohhh', 'stuff'

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
# from gensim.utils import simple_preprocess

# stop_words = stopwords.words('english')
# stop_words.extend(['from', 'subject', 're', 'edu', 'use'])

# def sent_to_words(sentences):
#     for sentence in sentences:
#         # deacc=True removes punctuations
#         yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))

# def remove_stopwords(texts):
#     return [[word for word in simple_preprocess(str(doc))
#              if word not in stop_words] for doc in texts]


# data_words = list(sent_to_words(texts))
# # remove stop words
# data_words = remove_stopwords(data_words)
# print(data_words[:1][0][:30])

In [ ]:
import gensim.corpora as corpora
# Create Dictionary
id2word = corpora.Dictionary(tokens_by_sentence)

# # # Create Corpus
texts = tokens_by_sentence

# # # Term Document Frequency
corpus = [id2word.doc2bow(text) for text in texts]

# # View
print(corpus[:1][0][:30])

[(0, 1), (1, 1), (2, 1)]


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
tokens_by_sentence[1]

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


['crust', 'good']

In [ ]:
id2word[3]

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


'crust'

In [ ]:

num_topics = 10
# Build LDA model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                       id2word=id2word,
                                       num_topics=num_topics)
# Print the Keyword in the 10 topics
print(lda_model.print_topics())
doc_lda = lda_model[corpus]

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


[(0, '0.025*"food" + 0.023*"place" + 0.019*"good" + 0.012*"really" + 0.010*"delicious" + 0.010*"eat" + 0.009*"staff" + 0.007*"well" + 0.007*"wait" + 0.007*"go"'), (1, '0.047*"good" + 0.024*"place" + 0.020*"back" + 0.016*"steak" + 0.015*"go" + 0.015*"service" + 0.014*"restaurant" + 0.011*"food" + 0.010*"ever" + 0.010*"recommend"'), (2, '0.045*"food" + 0.024*"good" + 0.021*"great" + 0.020*"service" + 0.016*"star" + 0.014*"give" + 0.011*"place" + 0.010*"eat" + 0.009*"like" + 0.008*"say"'), (3, '0.033*"food" + 0.012*"get" + 0.012*"place" + 0.012*"service" + 0.012*"go" + 0.011*"come" + 0.011*"bad" + 0.010*"minute" + 0.009*"experience" + 0.009*"good"'), (4, '0.022*"back" + 0.020*"good" + 0.019*"time" + 0.014*"service" + 0.014*"really" + 0.013*"go" + 0.011*"get" + 0.011*"great" + 0.011*"place" + 0.009*"like"'), (5, '0.026*"food" + 0.016*"great" + 0.015*"amazing" + 0.015*"place" + 0.015*"service" + 0.012*"go" + 0.012*"like" + 0.010*"say" + 0.010*"nice" + 0.009*"time"'), (6, '0.029*"good" + 0.0

In [ ]:
lda_model.print_topics(num_topics=6, num_words=5)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


[(4,
  '0.022*"back" + 0.020*"good" + 0.019*"time" + 0.014*"service" + 0.014*"really"'),
 (3,
  '0.033*"food" + 0.012*"get" + 0.012*"place" + 0.012*"service" + 0.012*"go"'),
 (9,
  '0.028*"great" + 0.020*"go" + 0.018*"place" + 0.015*"friendly" + 0.015*"wait"'),
 (1,
  '0.047*"good" + 0.024*"place" + 0.020*"back" + 0.016*"steak" + 0.015*"go"'),
 (8,
  '0.013*"place" + 0.011*"order" + 0.011*"good" + 0.009*"day" + 0.009*"service"'),
 (2,
  '0.045*"food" + 0.024*"good" + 0.021*"great" + 0.020*"service" + 0.016*"star"')]

In [ ]:
import pyLDAvis.gensim_models as gensimvis

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, id2word)
vis

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
3      0.058885 -0.015159       1        1  13.061697
2      0.094410 -0.050804       2        1  11.817480
1     -0.070441 -0.033262       3        1  10.354567
6     -0.019248  0.015097       4        1  10.188284
4     -0.073860 -0.026277       5        1   9.997875
9      0.000710 -0.053382       6        1   9.753704
5      0.038103 -0.042863       7        1   9.539803
0      0.056666  0.032502       8        1   8.977198
7     -0.101973  0.011718       9        1   8.971350
8      0.016747  0.162428      10        1   7.338043, topic_info=       Term        Freq       Total Category  logprob  loglift
4      good   94.000000   94.000000  Default  30.0000  30.0000
82     food  100.000000  100.000000  Default  29.0000  29.0000
43     back   47.000000   47.000000  Default  28.0000  28.0000
17    great   56.000000   56.000000  Default  27.0000  27.0000
407   steak   14.000000   14.000000  Default  26.0000  26.0000
..      ...         ...         ...      ...      ...      ...
18     menu    1.718922   13.152456  Topic10  -5.4563   0.5772
282  really    1.684193   28.216092  Topic10  -5.4768  -0.2065
145    come    1.495098   30.279750  Topic10  -5.5958  -0.3962
97    never    1.495066   20.770577  Topic10  -5.5959  -0.0193
249   think    1.495058   16.955154  Topic10  -5.5959   0.1837

[748 rows x 6 columns], token_table=      Topic      Freq        Term
term                             
1038      8  0.672194           I
223       1  0.281875  absolutely
223       2  0.281875  absolutely
223       3  0.281875  absolutely
223       7  0.140937  absolutely
...     ...       ...         ...
338       7  0.259942       yummy
338       9  0.519884       yummy
546       2  0.630733        zero
546       9  0.210244        zero
546      10  0.210244        zero

[1463 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[4, 3, 2, 7, 5, 10, 6, 1, 8, 9])

In [ ]:
count = 0
for i in lda_model[corpus]:
  print(f'Doc:{count}:{i}')
  count+=1
  if count ==10:
    break

Doc:0:[(0, 0.025003199), (1, 0.025013749), (2, 0.02500536), (3, 0.77494895), (4, 0.025003944), (5, 0.02500354), (6, 0.02500658), (7, 0.02500717), (8, 0.02500401), (9, 0.025003506)]
Doc:1:[(0, 0.03333655), (1, 0.033345588), (2, 0.033337634), (3, 0.033336494), (4, 0.033339974), (5, 0.03333616), (6, 0.03334546), (7, 0.6999462), (8, 0.03333962), (9, 0.033336308)]
Doc:2:[(0, 0.02500584), (1, 0.02500392), (2, 0.025002778), (3, 0.02501399), (4, 0.77494866), (5, 0.02501141), (6, 0.025002778), (7, 0.025004067), (8, 0.025003817), (9, 0.025002778)]
Doc:3:[(0, 0.010003283), (1, 0.010001337), (2, 0.010003555), (3, 0.90998167), (4, 0.010002039), (5, 0.010001674), (6, 0.010001331), (7, 0.010001504), (8, 0.010001972), (9, 0.01000165)]
Doc:4:[(0, 0.020002054), (1, 0.020010047), (2, 0.020005358), (3, 0.81996804), (4, 0.020000728), (5, 0.020002304), (6, 0.020001972), (7, 0.02000335), (8, 0.020003686), (9, 0.020002441)]
Doc:5:[(0, 0.01667111), (1, 0.016670771), (2, 0.8499677), (3, 0.016669922), (4, 0.0166

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


### BERTopic

In [ ]:
!pip install bertopic

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 48.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 10.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached Cython-0.29.36-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_24_x86_64.whl (1.9 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 51.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.8 MB/s eta 0:00:00
  Created wheel for hdbscan: filename=hdbscan-0.8.33-cp310-cp310-linux_x86_64.whl size=3039164 sha256=507c1ed2a04c8662fce2bbf847793cc5f0908488abe5e281526486068f8688ff
  Stored in dir

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
model_embedding = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embbeddings = model_embedding.encode(raw_text)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
corpus_embbeddings.shape

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


(1000, 384)

In [ ]:
model = BERTopic(
    n_gram_range=(1, 2),
    vectorizer_model="vectorizer",
    nr_topics='auto',
    min_topic_size=10,
    seed_topic_list=[
        ["experience", "bad", "good", "nice"],
        ["place", "atmosphere", "toilet", "clean"],
        ["staff", "waitress", "service"],
        ["wait", "time", "long"],
        ["food", "taste"]
    ],
    calculate_probabilities=True).fit(raw_text, corpus_embbeddings)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/hdbscan/hdbscan_.py:1170: DeprecationWarning: `alltrue` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `all` instead.
  self._all_finite = is_finite(X)


AttributeError: ignored